### 1. 택배 하차 - 삼성

In [1]:
import sys
from io import StringIO
from collections import deque

sys.stdin=StringIO("""\
10 10
5 8 2 7
12 9 1 10
13 1 2 4
14 10 1 6
19 1 1 9
3 7 5 1
6 1 1 4
8 2 2 2
9 1 1 8
1 1 1 8
""")
input=sys.stdin.readline

n, m = map(int, input().split())
space = [[0]*n for _ in range(n)]
boxes = {}
ramained_ids=[]

#중력 적용 함수
def falling(k):
    while True:
        can_fall = True
        r, c, w, h = boxes[k]
        if r+h>=n:  #더 떨어질 곳 x
            can_fall = False
        else:
            for nc in range(c, c+w):  #내려갈 자리가 모두 0인지 확인
                if space[r+h][nc]!=0:
                    can_fall=False
                    break

        if can_fall==False:
            break

        #박스 위치 업데이트
        for nc in range(c, c+w):
            space[r][nc] = 0
            space[r+h][nc]=k 

        boxes[k][0]+=1

#박스 투입
for i in range(m):
    k, h, w, c = map(int, input().split()) #택배번호, 세로크기, 가로크기, 좌측좌표
    r, c = 0, c-1  #좌상단 좌표
    boxes[k] = [r, c, w, h]  #좌상단x, 좌상단y, 가로, 세로
    ramained_ids.append(k)
    #박스 위치 업데이트
    for nr in range(r, r+h):
        for nc in range(c, c+w):
            space[nr][nc]=k
    #중력적용
    falling(k)

def finding(dir):
    can_out = []
    for k in ramained_ids:
        r, c, w, h = boxes[k]
        outable = True
        for nr in range(r, r+h):  #각 박스의 모든 row
            if dir =='left':  #왼쪽 비었는지 확인
                if sum(space[nr][0:c])>0: 
                    outable=False
                    break
            else:  #오른쪽 비었는지 확인
                if sum(space[nr][c+w:])>0: 
                    outable=False
                    break
        if outable==True:
            can_out.append(k)
    can_out.sort()
    return can_out

def delete(k):
    r, c, w, h = boxes[k]
    for nr in range(r, r+h):
        for nc in range(c, c+w):
            space[nr][nc]=0
    del boxes[k]
    ramained_ids.remove(k)

while ramained_ids:
    #왼쪽 검사
    todelete = finding('left')
    if todelete:
        print(todelete[0])
        delete(todelete[0])
        tofalling = sorted(ramained_ids, key=lambda x:boxes[x][0]+boxes[x][3], reverse=True)
        for k in tofalling:
            falling(k)
    #오른쪽 검사
    todelete = finding('right')
    if todelete:
        print(todelete[0])
        delete(todelete[0])
        tofalling = sorted(ramained_ids, key=lambda x:boxes[x][0]+boxes[x][3], reverse=True)
        for k in tofalling:
            falling(k)

3
1
8
12
6
9
13
19
14
5


### 2. 고대 문명 유적 탐사 - 삼성

In [ ]:
import sys
from io import StringIO
from collections import deque
import copy

input=sys.stdin.readline

k, m = map(int, input().split())
grid = []
for _ in range(5):
    nums = list(map(int, input().split()))
    grid.append(nums)
new = list(map(int, input().split()))

dx=[-1,-1,-1,0,1,1,1,0]
dy=[-1,0,1,1,1,0,-1,-1]

def get_hide(grid):
    visited=[[False]*5 for _ in range(5)]
    hide=[]
    for i in range(5):
        for j in range(5):
            if visited[i][j]==False:
                q = deque([(i,j)])
                visited[i][j]=True
                same=[(j,-i)]
                while q:
                    x, y = q.popleft()
                    for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                        nx, ny = x+dx, y+dy
                        if 0<=nx<5 and 0<=ny<5 and grid[x][y]==grid[nx][ny] and visited[nx][ny]==False:
                            q.append((nx,ny))
                            visited[nx][ny]=True
                            same.append((ny,-nx))
                if len(same)>=3:
                    hide+=same
    return list(set(hide))

def rotate_grid(grid, x, y, ang):
    new_grid = copy.deepcopy(grid)
    q = deque()
    for d in range(8):
        nx, ny = x+dx[d], y+dy[d]
        q.append(grid[nx][ny])

    if ang==90:
        q.rotate(2)
    elif ang==180:
        q.rotate(4)
    else:
        q.rotate(6)
    
    for d in range(8):
        nx, ny = x+dx[d], y+dy[d]
        new_grid[nx][ny]=q[d]
    return new_grid

def find_torotate(grid):
    info = []
    for x in range(1,4):
        for y in range(1,4):
            for ang in [90,180,270]:
                new_grid = rotate_grid(grid, x,y,ang)
                hide =  get_hide(new_grid)
                if not hide:
                    continue
                info.append((len(hide), -ang, -y, -x)) 
    sorted_info = sorted(info, reverse=True)
    return (-sorted_info[0][3], -sorted_info[0][2], -sorted_info[0][1]) if info else None


for _ in range(k):
    result = 0
    find_result =find_torotate(grid)
    if not find_result:
        break
    x, y, ang = find_result
    grid = rotate_grid(grid, x,y,ang)
    hide = get_hide(grid)
    result+=len(hide)
    sorted_hide = sorted(hide)
    for idx, (c, r) in enumerate(sorted_hide):
        grid[-r][c]=new[idx]
    new = new[idx+1:]

    while True:
        hide = get_hide(grid)
        if hide:
            result+=len(hide)
            sorted_hide = sorted(hide)
            for idx, (c, r) in enumerate(sorted_hide):
                grid[-r][c]=new[idx]
            new = new[idx+1:]
        else:
            break

    print(result, end=' ')

### 회전 공식

In [25]:
#90도 회전
def rotate_array_90(array):
    rotated_array=[]
    for row in zip(*array[::-1]):
        rotated_array.append(list(row))
    return rotated_array

#180도 회전
def rotate_array_180(array):
    rotated_array=[]
    for row in array[::-1]:
        rotated_array.append(row[::-1])
    return rotated_array

#270도 회전
def rotate_array_270(array):
    rotated_array=[]
    for row in list(zip(*array))[::-1]:
        rotated_array.append(list(row))
    return rotated_array

#전치 Transpose
def transpose_array(array):
    trasposed_array=[]
    for row in zip(*array):
        trasposed_array.append(list(row))
    return trasposed_array

#한번에 회전하는 방법 (90도 -> 원하는 각도만큼 반복)
def rotate(arr, si, sj):  #si, sj : array 시작점
    narr = [x[:] for x in arr]
    for i in range(3):  #3*3 array일 때
        for j in range(3):
            narr[si+i][sj+j]=arr[si+3-j-1][sj+i]
    return narr

In [ ]:
# 공식 적용한 풀이
import sys
from io import StringIO
from collections import deque

sys.stdin=StringIO("""\
2 20
7 6 7 6 7
6 7 6 7 6
6 7 1 5 4
7 6 3 2 1
5 4 3 2 7
3 2 3 5 2 4 6 1 3 2 5 6 2 1 5 6 7 1 2 3
""")
input=sys.stdin.readline

k, m = map(int, input().split())
grid = []
for _ in range(5):
    nums = list(map(int, input().split()))
    grid.append(nums)
new = deque(map(int, input().split()))

dx=[-1,-1,-1,0,1,1,1,0]
dy=[-1,0,1,1,1,0,-1,-1]

def get_hide(grid):
    visited=[[False]*5 for _ in range(5)]
    hide=[]
    for i in range(5):
        for j in range(5):
            if visited[i][j]==False:
                q = deque([(i,j)])
                visited[i][j]=True
                same=[(j,-i)]
                while q:
                    x, y = q.popleft()
                    for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                        nx, ny = x+dx, y+dy
                        if 0<=nx<5 and 0<=ny<5 and grid[x][y]==grid[nx][ny] and visited[nx][ny]==False:
                            q.append((nx,ny))
                            visited[nx][ny]=True
                            same.append((ny,-nx))
                if len(same)>=3:
                    hide+=same
    return list(hide)

def rotate(grid, si, sj):
    rotated = [row[:] for row in grid]
    for i in range(3):
        for j in range(3):
            rotated[si+i][sj+j]=grid[si+3-j-1][sj+i]
    return rotated

for _ in range(k):
    mx_cnt=0
    for rot in range(1,4): #회전수 
        for sj in range(0,3): #열
            for si in range(0,3): #행
                new_grid=[row[:] for row in grid]
                for _ in range(rot):
                    new_grid = rotate(new_grid, si, sj)
                hide =  get_hide(new_grid)
                if len(hide)>mx_cnt:
                    mx_cnt=len(hide)
                    best_grid=new_grid
    if mx_cnt==0:
        break

    result = 0
    grid = best_grid

    while True:
        hide = get_hide(grid)
        if hide:
            result+=len(hide)
            sorted_hide = sorted(hide)
            for idx, (c, r) in enumerate(sorted_hide):
                grid[-r][c]=new.popleft()
        else:
            break

    print(result, end=' ')

17 